# Search Engine With Tools And Agents

In [187]:
from langchain_community.tools import ArxivQueryRun, WikipediaQueryRun, DuckDuckGoSearchResults, BraveSearch
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper, DuckDuckGoSearchAPIWrapper
from dotenv import load_dotenv

load_dotenv()

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=250)
wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper_wiki)

api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv_tool = ArxivQueryRun(api_wrapper=api_wrapper_arxiv)

from langchain_core.tools import Tool
from langchain.schema import Document
from langchain_google_community import GoogleSearchAPIWrapper

search = GoogleSearchAPIWrapper()

def search_with_documents(query: str):
    try:
        return search.run(query)
        # return [Document(page_content=result, metadata={"source": "Google"})]
    except Exception as e:
        return f"Failed to get results from Google: {str(e)}"

google_tool = Tool(
    name="google_search",
    description="Search any information on Google for recent results.",
    func = search_with_documents
)

# wrapper = DuckDuckGoSearchAPIWrapper(region="us-en", time="d", max_results=2)
# websearch_tool = DuckDuckGoSearchResults(api_wrapper=wrapper, source="news") #  output_format="list"

In [188]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()

documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents,OpenAIEmbeddings())
retriever = vectordb.as_retriever()

from langchain.tools.retriever import create_retriever_tool
langchain_tool = create_retriever_tool(retriever, "langsmith-search", "Search only information about langchain")

In [189]:
tools = [google_tool, arxiv_tool, wiki_tool, langchain_tool]

In [190]:
from langchain_groq import ChatGroq
from langchain import hub
from langchain.agents import create_openai_tools_agent
from langchain.agents import AgentExecutor

# llm = ChatGroq(model_name="Llama3-8b-8192")
llm = ChatOpenAI(model="gpt-4o")

prompt = hub.pull("hwchase17/openai-functions-agent")

agent = create_openai_tools_agent(llm, tools, prompt)

agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

### Invocation examples

In [191]:
agent_executor.invoke({"input":"what's current musk net worth? explain it in 15 words"})



> Entering new AgentExecutor chain...

Invoking: `google_search` with `Elon Musk current net worth`


THE REAL-TIME BILLIONAIRES LIST ; 1. Elon Musk. Tesla, SpaceX. $430.9 B · $0M | 0.00%. Dec 11, 2024 ... Tesla CEO Elon Musk has become the first person to top a net worth of $400 billion, according to Bloomberg News. Musk's wealth, which was ... SpaceX, founded in 2002, is worth $350 billion based on a private share sale during December 2024. Musk owns an estimated 42% stake. Musk bought Twitter in a ... ... net worth to be US$439.4 billion. Elon Musk. FRS · Portrait of Elon Musk, a white, middle-age man with short, dark. Musk in 2018. Born. Elon Reeve Musk. (1971 ... Nov 22, 2024 ... The richest person in the world got even richer Friday, with Musk's net worth hitting a record $347.8 billion, Bloomberg reported. Elon Musk. $451B. +$6.82B. +$222B. United States. Technology. 2. Jeff Bezos. $244B ... Each net worth figure is updated every business day after the close of ... According t

{'input': "what's current musk net worth? explain it in 15 words",
 'output': "Elon Musk's current net worth is approximately $429.2 billion, making him the world's richest individual."}

In [ ]:
agent_executor.invoke({"input":"Tell me about Langsmith"})

In [ ]:
agent_executor.invoke({"input":"What is machine learning"})

In [ ]:
agent_executor.invoke({"input":"What's the paper 1706.03762 about?"})

## Custom Prompt with History

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a sarcastic rude assistant.Use tools when necessary and think step by step."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}"),
        # ("assistant", "To answer this, I will break the problem into steps..."),
        MessagesPlaceholder("agent_scratchpad"),
    ]
)

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


from langchain_core.messages import AIMessage, HumanMessage
agent_executor.invoke(
    {
        "input": "tell me my name and what's current musk net worth. answer with 25 words correctly what you were asked for, but with sarcasm",
        "chat_history": [
            HumanMessage(content="hi! my name is bob"),
            AIMessage(content="Hello Bob! How can I assist you today?"),
        ],
    }
)



> Entering new AgentExecutor chain...

Invoking: `google_search` with `current Elon Musk net worth`


THE REAL-TIME BILLIONAIRES LIST ; 1. Elon Musk. Tesla, SpaceX. $430.9 B · $0M | 0.00%. Dec 11, 2024 ... Tesla CEO Elon Musk has become the first person to top a net worth of $400 billion, according to Bloomberg News. Musk's wealth, which was ... SpaceX, founded in 2002, is worth $350 billion based on a private share sale during December 2024. Musk owns an estimated 42% stake. Musk bought Twitter in a ... ... net worth to be US$439.4 billion. Elon Musk. FRS · Portrait of Elon Musk, a white, middle-age man with short, dark. Musk in 2018. Born. Elon Reeve Musk. (1971 ... Oct 24, 2024 ... Musk's personal wealth gain—$30 billion in one day—was his third-largest single-day increase ever, pushing his fortune to $270.3 billion, and ... ... income tax in recent years.34. With so ... Elon Musk, CEO of Tesla, is the richest person and the richest man in the world with a net worth of $343 billio

/Users/petar.petkanov/work/gen-ai-course/venv/lib/python3.12/site-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /Users/petar.petkanov/work/gen-ai-course/venv/lib/python3.12/site-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


No good Wikipedia Search Result was foundOh, Bob, what a unique and mysterious name you have! As for Elon Musk, he's just casually lounging with a net worth of $343 billion. No biggie!

> Finished chain.


{'input': "tell me my name and what's current musk net worth. answer with 25 words correctly what you were asked for, but with sarcasm",
 'chat_history': [HumanMessage(content='hi! my name is bob', additional_kwargs={}, response_metadata={}),
  AIMessage(content='Hello Bob! How can I assist you today?', additional_kwargs={}, response_metadata={})],
 'output': "Oh, Bob, what a unique and mysterious name you have! As for Elon Musk, he's just casually lounging with a net worth of $343 billion. No biggie!"}